# EduAssist AI — LLMOps: API Cost, Latency and Prompt Version Management

**Course:** MLOps (AIMLCZG523) · **Experiential Learning Assignment 2** · **Student:** Sivaram Gudlavalleti (2025ae05220)

This notebook is the executable deliverable for all three tasks. It is run end-to-end on the
**BITS MLOps lab VM** by `scripts/run_pipeline.py` (see `LAB_RUNBOOK.md`), which records the host,
Python, package versions and git commit of the run in `reports/run/run_summary.md`.

| Section | Task | Marks | Main outputs |
|---|---|---|---|
| 0 | Setup: environment, secrets, configuration, pricing, questions, prompts | – | – |
| 1 | Call an LLM and measure latency | 2 | latency table, summary statistics, slowest-request analysis |
| 2 | Count tokens and calculate cost | 3 | `outputs/latency_tokens_cost.csv`, monthly projection, optimisation recommendation |
| 3 | Prompt versioning and MLflow evaluation | 5 | MLflow experiment `eduassist_llmops_assignment`, Prompt Registry v1–v3, `outputs/prompt_evaluation_results.csv`, `production` alias |

### Phase-wise approach

![Phase plan](../docs/phase_plan.png)

## LLMOps vs MLOps (learning outcome 1)

**MLOps** manages the lifecycle of a model *we train*: datasets, feature pipelines, training runs, model
registry, deployment and drift monitoring. **LLMOps** manages an application built on a model *we usually
do not train* (a hosted foundation model reached through an API), so the versioned production artefacts shift:

| Concern | MLOps (Assignment 1) | LLMOps (this assignment) |
|---|---|---|
| Main artefact that changes | model weights from a training run | prompt template, model/API version, retrieved context, guardrails |
| Versioning | data + code + model registry | **Prompt Registry** (versions, commit messages, diffs, aliases) + model/API version |
| Cost driver | training compute, fixed serving infrastructure | **per-request tokens** (input + output, priced per million) |
| Latency driver | model size, hardware | prompt length, **generated output length**, provider load, network |
| Quality checks | accuracy / ROC-AUC on a held-out test set | output-format validity (`valid_json`), content checks, human review — outputs are non-deterministic |
| Promotion | best validation metric → model registry stage | evaluated prompt version → registry **alias** (`production`) |

## 0. Setup

In [ ]:
# =============================================================================
# Section 0.1 - Imports
# Standard-library modules handle timing, JSON parsing, prompt diffs and files;
# pandas/numpy build the result tables, matplotlib draws the charts, MLflow 3.x
# provides experiment tracking + the Prompt Registry, and python-dotenv loads the
# API key from a git-ignored .env file so it is never hard-coded.
# =============================================================================
import difflib            # unified diff between prompt versions (Task 3 evidence)
import importlib.metadata  # package versions for the reproducibility record
import json               # parse and validate model output (valid_json metric)
import os                 # read configuration and secrets from environment variables
import platform           # Python / OS details for the run record
import socket             # host name -> proof that the run happened on the lab VM
import time               # perf_counter() for latency, sleep() for rate limiting
from datetime import datetime, timezone
from pathlib import Path

import matplotlib

matplotlib.use("Agg")      # headless backend: works on the lab VM and under nbconvert
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display
from mlflow.genai.scorers import scorer

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

In [ ]:
# =============================================================================
# Section 0.2 - Project paths, secrets and run environment
# The notebook may be executed from notebooks/ (Jupyter) or the repo root
# (nbconvert via run_pipeline.py), so the project root is located by walking up
# to the folder that contains requirements.txt. Secrets are read from .env.
# =============================================================================
def find_project_root(start: Path) -> Path:
    """Return the first folder at or above `start` that contains requirements.txt."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("requirements.txt not found - run the notebook inside the project folder")


ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = ROOT / "outputs"            # CSV deliverables
FIG_DIR = OUTPUT_DIR / "figures"         # charts used in the report
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# load_dotenv copies KEY=value pairs from .env into os.environ; existing environment
# variables win (override=False), so `export GEMINI_API_KEY=...` also works.
load_dotenv(ROOT / ".env", override=False)
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")   # silence MLflow's IDE hint banner


def pkg_version(name: str) -> str:
    """Installed version of a package, or 'not installed'."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


# Record where and with what this notebook ran (shown in the report as execution evidence)
RUN_ENV = {
    "host": socket.gethostname(),
    "os": f"{platform.system()} {platform.release()}",
    "python": platform.python_version(),
    "run_started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    **{f"pkg_{p}": pkg_version(p) for p in ["mlflow", "google-genai", "openai", "anthropic", "pandas"]},
}
pd.Series(RUN_ENV, name="value").to_frame()

In [ ]:
# =============================================================================
# Section 0.3 - Run configuration (provider, model, generation settings)
# Everything is driven by environment variables with safe defaults, so the same
# notebook runs unchanged on any machine. Default = Google Gemini 2.5 Flash on the
# free tier. Only the *presence* of the API key is printed - never its value.
# =============================================================================
PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()          # gemini | openai | anthropic | ollama
DEFAULT_MODELS = {"gemini": "gemini-2.5-flash", "openai": "gpt-4o-mini",
                  "anthropic": "claude-haiku-5-5", "ollama": "qwen2.5-coder:7b"}
MODEL = os.getenv("LLM_MODEL", DEFAULT_MODELS[PROVIDER])
API_KEY_VARS = {"gemini": ["GEMINI_API_KEY", "GOOGLE_API_KEY"], "openai": ["OPENAI_API_KEY"],
                "anthropic": ["ANTHROPIC_API_KEY"], "ollama": []}

# API version used by each SDK (documented in the report as required by section 7)
API_VERSION = {"gemini": "Gemini API v1beta (google-genai SDK)", "openai": "OpenAI Chat Completions v1",
               "anthropic": "Anthropic Messages API 2023-06-01", "ollama": "Ollama OpenAI-compatible /v1"}[PROVIDER]

TEMPERATURE = 0.0          # minimise randomness so prompt versions are compared fairly
MAX_OUTPUT_TOKENS = 2048   # hard cap on generated tokens = cap on cost and latency per request
THINKING_BUDGET = 0        # Gemini 2.5 Flash: disable "thinking" tokens -> billed tokens = visible answer
REQUEST_INTERVAL_S = float(os.getenv("REQUEST_INTERVAL_SECONDS", "0" if PROVIDER == "ollama" else "6"))
MAX_ATTEMPTS = 5           # retries for rate-limit (429) / transient server errors (5xx)
MONTHLY_REQUESTS = 10_000  # volume used for the monthly cost projection (Task 2)


def api_key_for(provider: str) -> str | None:
    """Return the first API-key environment variable that is set for `provider` (None for Ollama)."""
    for var in API_KEY_VARS[provider]:
        if os.getenv(var):
            return os.environ[var]
    return None


if PROVIDER != "ollama" and not api_key_for(PROVIDER):
    raise RuntimeError(f"Set one of {API_KEY_VARS[PROVIDER]} in the environment or in .env (see .env.example)")

pd.Series({"provider": PROVIDER, "model": MODEL, "api_version": API_VERSION, "temperature": TEMPERATURE,
           "max_output_tokens": MAX_OUTPUT_TOKENS, "request_interval_s": REQUEST_INTERVAL_S,
           "api_key": "set (hidden)" if api_key_for(PROVIDER) else "not required"}, name="value").to_frame()

In [ ]:
# =============================================================================
# Section 0.4 - Published pricing (USD per 1 million tokens)
# Source: https://ai.google.dev/gemini-api/docs/pricing (paid "Standard" tier, text),
# page last updated 2026-10-28. Thinking tokens are billed at the output rate.
# The free tier used for the calls bills $0; costs below are what the same traffic
# would cost on the paid tier, which is the number a production budget needs.
# Any other model can be priced with LLM_INPUT_PRICE_PER_1M / LLM_OUTPUT_PRICE_PER_1M.
# =============================================================================
PRICING_DATE = "2026-10-28"
CURRENCY = "USD"
PRICING_SOURCE = "https://ai.google.dev/gemini-api/docs/pricing"
PRICING_USD_PER_1M = {
    "gemini-2.5-flash":      {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "gemini-2.5-pro":        {"input": 1.25, "output": 10.00},
}
if os.getenv("LLM_INPUT_PRICE_PER_1M"):          # manual override for non-Gemini models
    PRICING_USD_PER_1M[MODEL] = {"input": float(os.environ["LLM_INPUT_PRICE_PER_1M"]),
                                 "output": float(os.environ["LLM_OUTPUT_PRICE_PER_1M"])}
elif PROVIDER == "ollama":
    # Local open-source model (brief, section 7): no API bill, so the cost is *estimated* with the
    # published hosted price of a comparable small model (Gemini 2.5 Flash-Lite).
    PRICING_USD_PER_1M.setdefault(MODEL, PRICING_USD_PER_1M["gemini-2.5-flash-lite"])

PRICE = PRICING_USD_PER_1M[MODEL]                 # rates for the model actually called
pd.DataFrame(PRICING_USD_PER_1M).T.rename(columns={"input": "input $/1M", "output": "output $/1M"})

In [ ]:
# =============================================================================
# Section 0.5 - The five academic questions (same set for every task and prompt)
# Each question carries `expected_keywords` - concepts a correct answer should
# mention ("a|b" = either spelling). They give a cheap, deterministic content
# quality score (keyword coverage) alongside the format check in Task 3.
# =============================================================================
QUESTIONS = json.loads((ROOT / "data" / "questions.json").read_text(encoding="utf-8"))
questions_df = pd.DataFrame(QUESTIONS)
questions_df[["question_id", "subject", "question"]]

In [ ]:
# =============================================================================
# Section 0.6 - Prompt templates v1, v2, v3 (exact wording from the brief)
# {{question}} is MLflow's double-brace template variable. The same text is
# registered in the Prompt Registry in Task 3; Task 1 uses v1 directly.
# =============================================================================
PROMPT_NAME = "eduassist-academic-qa"     # Prompt Registry entry that holds all versions

PROMPTS = {
    "v1": {
        "style": "basic",
        "commit_message": "v1 basic: minimal instruction, free-text answer (baseline)",
        "template": "Answer the following student question clearly.\nQuestion: {{question}}",
    },
    "v2": {
        "style": "structured",
        "commit_message": "v2 structured: academic-assistant role, concise/accurate, undergraduate-level language",
        "template": (
            "You are an academic assistant.\n"
            "Answer the following student question in a clear, concise, and accurate manner.\n"
            "Use simple language suitable for undergraduate students.\n"
            "Question: {{question}}"
        ),
    },
    "v3": {
        "style": "json",
        "commit_message": "v3 json: strict JSON-only output with answer/confidence/key_points schema for the app",
        "template": (
            "You are an academic assistant.\n"
            "Answer the student question and return ONLY valid JSON.\n"
            "Do not include markdown, explanations outside JSON, or code fences.\n"
            "Use this exact JSON schema:\n"
            "{\n"
            '  "answer": "string",\n'
            '  "confidence": "high | medium | low",\n'
            '  "key_points": ["string"]\n'
            "}\n"
            "Question: {{question}}"
        ),
    },
}


def render(template: str, question: str) -> str:
    """Fill the {{question}} placeholder (same substitution MLflow's PromptVersion.format() does)."""
    return template.replace("{{question}}", question)


print(render(PROMPTS["v3"]["template"], QUESTIONS[0]["question"]))

In [ ]:
# =============================================================================
# Section 0.7 - LLM client: one API call -> one fully-measured record
# `call_llm()` is the single entry point used by every task. It
#   1. stamps wall-clock start/end times and measures latency with perf_counter
#      around the API call only (prompt building and parsing are excluded),
#   2. extracts token usage from the provider's response object,
#   3. retries rate-limit / transient errors with exponential back-off and records
#      the number of attempts (latency = the successful attempt), and
#   4. never raises: a failed request becomes a record with status="error".
# Provider adapters return (text, input_tokens, output_tokens, thinking_tokens,
# finish_reason, served_model_version) so the rest of the notebook is provider-neutral.
# =============================================================================
def make_client():
    """Create the SDK client for PROVIDER, reading the key from the environment."""
    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types
        return genai.Client(api_key=api_key_for("gemini"), http_options=types.HttpOptions(api_version="v1beta"))
    if PROVIDER == "anthropic":
        import anthropic
        return anthropic.Anthropic(api_key=api_key_for("anthropic"))
    from openai import OpenAI   # OpenAI itself, or any OpenAI-compatible server such as Ollama
    if PROVIDER == "ollama":
        return OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
    return OpenAI(api_key=api_key_for("openai"))


CLIENT = make_client()


def _call_gemini(prompt: str):
    """Gemini generate_content; usage_metadata holds prompt / candidates / thoughts token counts."""
    from google.genai import types
    config = types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_OUTPUT_TOKENS)
    if MODEL.startswith("gemini-2.5-flash"):   # thinking can only be switched off on 2.5 Flash models
        config.thinking_config = types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    resp = CLIENT.models.generate_content(model=MODEL, contents=prompt, config=config)
    usage = resp.usage_metadata
    finish = resp.candidates[0].finish_reason.name if resp.candidates and resp.candidates[0].finish_reason else ""
    return (resp.text or "", usage.prompt_token_count or 0, usage.candidates_token_count or 0,
            usage.thoughts_token_count or 0, finish, resp.model_version or MODEL)


def _call_openai_compatible(prompt: str):
    """Chat Completions (OpenAI / Ollama); usage.prompt_tokens and usage.completion_tokens."""
    resp = CLIENT.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}],
                                          temperature=TEMPERATURE, max_tokens=MAX_OUTPUT_TOKENS)
    return (resp.choices[0].message.content or "", resp.usage.prompt_tokens, resp.usage.completion_tokens, 0,
            resp.choices[0].finish_reason or "", resp.model)


def _call_anthropic(prompt: str):
    """Anthropic Messages API; usage.input_tokens and usage.output_tokens."""
    resp = CLIENT.messages.create(model=MODEL, max_tokens=MAX_OUTPUT_TOKENS, temperature=TEMPERATURE,
                                  messages=[{"role": "user", "content": prompt}])
    text = "".join(block.text for block in resp.content if block.type == "text")
    return text, resp.usage.input_tokens, resp.usage.output_tokens, 0, resp.stop_reason or "", resp.model


ADAPTERS = {"gemini": _call_gemini, "openai": _call_openai_compatible,
            "ollama": _call_openai_compatible, "anthropic": _call_anthropic}


def _http_status(exc: Exception) -> int | None:
    """HTTP status code carried by an SDK exception (attribute name differs per SDK)."""
    for attr in ("code", "status_code"):
        value = getattr(exc, attr, None)
        if isinstance(value, int):
            return value
    return None


def _is_retryable(exc: Exception) -> bool:
    """True for rate limits (429), server overload (5xx) and network errors - worth retrying."""
    status = _http_status(exc)
    return status in (408, 429, 500, 502, 503, 504) or (status is None and "connect" in type(exc).__name__.lower())


def call_llm(prompt: str, question_id: str, prompt_version: str) -> dict:
    """Send one prompt and return a record with timing, status, token usage and the response text."""
    record = {"question_id": question_id, "prompt_version": prompt_version, "provider": PROVIDER, "model": MODEL}
    for attempt in range(1, MAX_ATTEMPTS + 1):
        start_wall = datetime.now(timezone.utc)
        t0 = time.perf_counter()                       # monotonic high-resolution clock
        try:
            text, tin, tout, tthink, finish, served = ADAPTERS[PROVIDER](prompt)
            latency = time.perf_counter() - t0
            record.update(start_time=start_wall.isoformat(timespec="milliseconds"),
                          end_time=datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
                          latency_seconds=round(latency, 4), status="success", http_attempts=attempt,
                          input_tokens=tin, output_tokens=tout + tthink, thinking_tokens=tthink,
                          total_tokens=tin + tout + tthink, finish_reason=finish, served_model=served,
                          response_chars=len(text), response_text=text, error="")
            return record
        except Exception as exc:  # noqa: BLE001 - every failure must become a record, not a crash
            latency = time.perf_counter() - t0
            if attempt < MAX_ATTEMPTS and _is_retryable(exc):
                wait = min(10 * 2 ** (attempt - 1), 60)  # 10, 20, 40, 60 s back-off
                print(f"  {question_id}/{prompt_version}: HTTP {_http_status(exc)} - retry {attempt} in {wait}s")
                time.sleep(wait)
                continue
            record.update(start_time=start_wall.isoformat(timespec="milliseconds"),
                          end_time=datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
                          latency_seconds=round(latency, 4), status=f"error {_http_status(exc) or ''}".strip(),
                          http_attempts=attempt, input_tokens=0, output_tokens=0, thinking_tokens=0,
                          total_tokens=0, finish_reason="", served_model="", response_chars=0,
                          response_text="", error=f"{type(exc).__name__}: {str(exc)[:200]}")
            return record


def run_batch(prompt_version: str, template: str) -> pd.DataFrame:
    """Call the LLM once per question with `template`, pausing between calls to respect rate limits."""
    rows = []
    for i, q in enumerate(QUESTIONS):
        if i:
            time.sleep(REQUEST_INTERVAL_S)
        rec = call_llm(render(template, q["question"]), q["question_id"], prompt_version)
        print(f"  {rec['question_id']} {prompt_version}: {rec['status']:<8} {rec['latency_seconds']:6.2f}s "
              f"in={rec['input_tokens']:>4} out={rec['output_tokens']:>5}")
        rows.append(rec)
    return pd.DataFrame(rows)

## Task 1 — Call an LLM and measure latency (2 marks)

Each of the five questions is sent once with prompt **v1** (the baseline). For every request the record holds
question ID, prompt version, model name, start time, end time, latency and response status.

In [ ]:
# =============================================================================
# Task 1.1 - Execute the five API calls and build the latency table
# run_batch() calls call_llm() per question; the resulting DataFrame is the raw
# measurement log reused by Task 2 (tokens and cost come from the same calls).
# =============================================================================
print(f"Calling {PROVIDER}:{MODEL} for {len(QUESTIONS)} questions with prompt v1 ...")
task1_df = run_batch("v1", PROMPTS["v1"]["template"])

LATENCY_COLS = ["question_id", "prompt_version", "model", "start_time", "end_time", "latency_seconds", "status"]
assert (task1_df["status"] == "success").all(), "some requests failed - see the `error` column"
task1_df[LATENCY_COLS]

In [ ]:
# =============================================================================
# Task 1.2 - Latency summary statistics
# mean / median / min / max / standard deviation (sample std, ddof=1) over the
# five successful requests. Saved for the report as outputs/latency_summary.csv.
# =============================================================================
lat = task1_df.loc[task1_df["status"] == "success", "latency_seconds"]
latency_summary = pd.DataFrame({
    "statistic": ["count", "mean", "median", "min", "max", "std (sample)"],
    "latency_seconds": [len(lat), lat.mean(), lat.median(), lat.min(), lat.max(), lat.std(ddof=1)],
}).round(3)
latency_summary.to_csv(OUTPUT_DIR / "latency_summary.csv", index=False)
latency_summary

In [ ]:
# =============================================================================
# Task 1.3 - Latency chart: one bar per request with the mean as a reference line
# Output tokens are written on each bar because generated length is the main
# latency driver - the chart and the interpretation below make that visible.
# =============================================================================
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4df"           # text / secondary text / gridlines
SERIES = {"v1": "#2a78d6", "v2": "#eb6834", "v3": "#1baf7a"}  # fixed colour per prompt version


def style_axes(ax, title: str, ylabel: str) -> None:
    """Recessive axes: no top/right spines, light horizontal grid, titles in text ink."""
    ax.set_title(title, loc="left", fontsize=12, color=INK, fontweight="bold")
    ax.set_ylabel(ylabel, color=MUTED)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(task1_df["question_id"], task1_df["latency_seconds"], width=0.55, color=SERIES["v1"])
ax.axhline(lat.mean(), color=MUTED, linestyle="--", linewidth=1, label=f"mean {lat.mean():.2f}s")
ax.legend(frameon=False, loc="upper left")
for bar, (_, row) in zip(bars, task1_df.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{row.latency_seconds:.2f}s\n{row.output_tokens} out-tok",
            ha="center", va="bottom", fontsize=8.5, color=INK)
style_axes(ax, f"Task 1 - latency per request ({MODEL}, prompt v1)", "latency (s)")
ax.set_ylim(0, task1_df["latency_seconds"].max() * 1.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "task1_latency.png", dpi=150)
display(Image(str(FIG_DIR / "task1_latency.png")))

In [ ]:
# =============================================================================
# Task 1.4 - Slowest request: data-driven interpretation
# Compares the slowest request with the others on output tokens, input tokens and
# throughput, and checks whether latency tracks output length (Pearson r).
# The printed paragraph is the "short interpretation" deliverable.
# =============================================================================
ok = task1_df[task1_df["status"] == "success"].copy()
ok["output_tokens_per_s"] = ok["output_tokens"] / ok["latency_seconds"]
slow = ok.loc[ok["latency_seconds"].idxmax()]
others = ok.drop(index=slow.name)
r_out = ok["latency_seconds"].corr(ok["output_tokens"])
first_qid = ok.iloc[0]["question_id"]
q_text = questions_df.set_index("question_id").loc[slow["question_id"], "question"]

interpretation = (
    f"The slowest request was {slow['question_id']} (\"{q_text}\") at {slow['latency_seconds']:.2f}s, "
    f"{slow['latency_seconds'] / others['latency_seconds'].mean():.1f}x the mean of the other four "
    f"({others['latency_seconds'].mean():.2f}s). It generated {int(slow['output_tokens'])} output tokens versus an "
    f"average of {others['output_tokens'].mean():.0f} for the others, while its input was only "
    f"{int(slow['input_tokens'])} tokens. Across the five requests latency correlates with output length "
    f"(Pearson r = {r_out:.2f}): an LLM generates the answer token by token, so a question that invites a long, "
    f"multi-part answer (causes *and* consequences, several concepts) takes longer, whereas the prompt is processed "
    f"in parallel and short prompts add little. "
    + (f"{first_qid} was also the first call of the session, so it additionally paid for connection/TLS set-up. "
       if slow["question_id"] == first_qid else "")
    + "Other contributors outside our control are provider-side queueing and load on the shared free tier and "
    "network round-trip time from the lab VM; these explain residual variation between requests of similar length."
)
display(ok[["question_id", "latency_seconds", "input_tokens", "output_tokens", "output_tokens_per_s"]].round(2))
print(interpretation)

## Task 2 — Count tokens and calculate cost (3 marks)

Token usage comes from the API response (`usage_metadata` for Gemini). Cost uses the published per-million-token
rates from Section 0.4 (pricing date **2026-10-28**, currency **USD**):

* Input cost = input tokens / 1,000,000 × input price per million
* Output cost = output tokens / 1,000,000 × output price per million
* Total cost per request = input cost + output cost
* Projected monthly cost = average cost per request × 10,000

In [ ]:
# =============================================================================
# Task 2.1 - Cost functions and the per-request token/cost table
# add_costs() applies the assignment's formulas row by row; the result is saved
# as outputs/latency_tokens_cost.csv (latency + tokens + cost per request).
# =============================================================================
def input_cost(tokens: float, price=PRICE) -> float:
    """Input cost = input tokens / 1,000,000 x input price per million tokens."""
    return tokens / 1_000_000 * price["input"]


def output_cost(tokens: float, price=PRICE) -> float:
    """Output cost = output tokens / 1,000,000 x output price per million tokens."""
    return tokens / 1_000_000 * price["output"]


def add_costs(df: pd.DataFrame, price=PRICE) -> pd.DataFrame:
    """Return a copy of `df` with input_cost_usd, output_cost_usd and total_cost_usd columns."""
    out = df.copy()
    out["input_cost_usd"] = out["input_tokens"].map(lambda t: input_cost(t, price))
    out["output_cost_usd"] = out["output_tokens"].map(lambda t: output_cost(t, price))
    out["total_cost_usd"] = out["input_cost_usd"] + out["output_cost_usd"]
    return out


task2_df = add_costs(task1_df)
CSV_COLS = LATENCY_COLS + ["input_tokens", "output_tokens", "thinking_tokens", "total_tokens",
                           "input_cost_usd", "output_cost_usd", "total_cost_usd", "finish_reason", "http_attempts"]
task2_df[CSV_COLS].to_csv(OUTPUT_DIR / "latency_tokens_cost.csv", index=False)

cost_table = task2_df[["question_id", "input_tokens", "output_tokens", "total_tokens",
                       "input_cost_usd", "output_cost_usd", "total_cost_usd"]].copy()
totals = cost_table.sum(numeric_only=True)
cost_table.loc[len(cost_table)] = {"question_id": "TOTAL", **totals.to_dict()}
cost_table.style.format({c: "${:.6f}" for c in ["input_cost_usd", "output_cost_usd", "total_cost_usd"]})

In [ ]:
# =============================================================================
# Task 2.2 - Monthly cost projection for 10,000 requests
# Projected monthly cost = average cost per request x 10,000. The split shows
# which side (input or output tokens) drives the bill - it decides the optimisation.
# =============================================================================
avg_cost = task2_df["total_cost_usd"].mean()
monthly = avg_cost * MONTHLY_REQUESTS
output_share = task2_df["output_cost_usd"].sum() / task2_df["total_cost_usd"].sum()

projection = pd.Series({
    "model": MODEL,
    "pricing (USD / 1M tokens)": f"input ${PRICE['input']:.2f}, output ${PRICE['output']:.2f} (as of {PRICING_DATE})",
    "avg input tokens / request": round(task2_df["input_tokens"].mean(), 1),
    "avg output tokens / request": round(task2_df["output_tokens"].mean(), 1),
    "avg cost / request (USD)": f"${avg_cost:.6f}",
    "projected monthly cost, 10,000 requests (USD)": f"${monthly:.2f}",
    "share of cost from output tokens": f"{output_share:.1%}",
}, name="value").to_frame()
projection

In [ ]:
# =============================================================================
# Task 2.3 - Cost chart: input vs output cost per request (stacked bars)
# Values are shown in micro-dollars (millionths of a dollar) so the axis is readable.
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 4))
x = task2_df["question_id"]
inp = task2_df["input_cost_usd"] * 1e6
outc = task2_df["output_cost_usd"] * 1e6
ax.bar(x, inp, width=0.55, color="#2a78d6", label="input tokens", edgecolor="white", linewidth=1)
ax.bar(x, outc, width=0.55, bottom=inp, color="#eb6834", label="output tokens", edgecolor="white", linewidth=1)
for xi, total in zip(x, inp + outc):
    ax.text(xi, total, f"{total:.0f}", ha="center", va="bottom", fontsize=9, color=INK)
style_axes(ax, f"Task 2 - cost per request (micro-USD), avg ${avg_cost:.6f} -> ${monthly:.2f} / 10k requests",
           "cost (micro-USD)")
ax.legend(frameon=False, loc="upper left")
ax.set_ylim(0, (inp + outc).max() * 1.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "task2_cost_per_request.png", dpi=150)
display(Image(str(FIG_DIR / "task2_cost_per_request.png")))

In [ ]:
# =============================================================================
# Task 2.4 - Cost-optimisation what-if analysis and recommendation
# Re-prices the measured token counts under three levers, so the recommendation
# is backed by numbers: (a) shorter responses, (b) a smaller model for simple
# queries, (c) both. Context caching is shown for completeness - the prompts here
# are far below the minimum cacheable prefix, so it would not help this workload.
# =============================================================================
lite = PRICING_USD_PER_1M["gemini-2.5-flash-lite"]
avg_in, avg_out = task2_df["input_tokens"].mean(), task2_df["output_tokens"].mean()
OUTPUT_CAP = 250   # concise-answer target in tokens (e.g. via max_output_tokens + "answer in <= 150 words")


def monthly_cost(avg_input: float, avg_output: float, price) -> float:
    """Monthly cost for MONTHLY_REQUESTS requests at the given average token counts and prices."""
    return (input_cost(avg_input, price) + output_cost(avg_output, price)) * MONTHLY_REQUESTS


capped_out = task2_df["output_tokens"].clip(upper=OUTPUT_CAP).mean()
scenarios = pd.DataFrame([
    ("Current: " + MODEL, avg_in, avg_out, monthly_cost(avg_in, avg_out, PRICE)),
    (f"(a) Shorter responses (<= {OUTPUT_CAP} output tokens)", avg_in, capped_out, monthly_cost(avg_in, capped_out, PRICE)),
    ("(b) Route simple queries to gemini-2.5-flash-lite", avg_in, avg_out, monthly_cost(avg_in, avg_out, lite)),
    ("(c) Flash-Lite + shorter responses", avg_in, capped_out, monthly_cost(avg_in, capped_out, lite)),
], columns=["scenario", "avg_input_tokens", "avg_output_tokens", "monthly_cost_usd"])
scenarios["saving_vs_current"] = 1 - scenarios["monthly_cost_usd"] / scenarios.loc[0, "monthly_cost_usd"]
display(scenarios.style.format({"avg_input_tokens": "{:.0f}", "avg_output_tokens": "{:.0f}",
                                "monthly_cost_usd": "${:.2f}", "saving_vs_current": "{:.0%}"}))

recommendation = (
    f"Recommendation: output tokens are {output_share:.0%} of the bill because they cost "
    f"{PRICE['output'] / PRICE['input']:.1f}x more than input tokens and the answers are long "
    f"(avg {avg_out:.0f} output vs {avg_in:.0f} input tokens). The highest-impact lever is therefore to shorten "
    f"responses - a concise, structured format (as prompt v3 does) plus a max_output_tokens cap of about "
    f"{OUTPUT_CAP} would cut the projected monthly cost by {scenarios.loc[1, 'saving_vs_current']:.0%}, from "
    f"${scenarios.loc[0, 'monthly_cost_usd']:.2f} to ${scenarios.loc[1, 'monthly_cost_usd']:.2f}. Routing simple "
    f"factual questions to gemini-2.5-flash-lite saves {scenarios.loc[2, 'saving_vs_current']:.0%} on its own, and "
    f"{scenarios.loc[3, 'saving_vs_current']:.0%} combined with shorter answers. Reducing context size or caching "
    f"the stable prompt prefix brings little here because input is a small share of cost."
)
print(recommendation)

## Task 3 — Prompt versioning and MLflow evaluation (5 marks)

1. Create the MLflow experiment `eduassist_llmops_assignment` (SQLite tracking store `mlflow.db`, artifacts in `mlruns/`).
2. Register v1, v2, v3 in the **MLflow Prompt Registry** under `eduassist-academic-qa`, each with a commit message.
3. Run every version on the same five questions — one MLflow run per version, the prompt loaded *from the registry*
   (which links the run to the prompt version) — and log `avg_latency_seconds`, `avg_input_tokens`,
   `avg_output_tokens`, `avg_cost_per_request` and `valid_json`.
4. Score the responses with `mlflow.genai.evaluate()` and custom scorers; `valid_json` is a boolean per response,
   which MLflow aggregates into a mean = **Valid JSON responses / Total responses**.
5. Compare the versions and promote the best one with the alias **`production`**.

In [ ]:
# =============================================================================
# Task 3.1 - MLflow tracking setup and experiment creation
# A SQLite backend is required for the Prompt Registry. The experiment's artifact
# location is pinned to <repo>/mlruns so `mlflow ui` started from the repo root
# finds the artifacts regardless of the notebook's working directory.
# =============================================================================
EXPERIMENT_NAME = "eduassist_llmops_assignment"
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_tracking_uri(TRACKING_URI)

if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=(ROOT / "mlruns").as_uri(),
                             tags={"project": "EduAssist AI", "assignment": "MLOps A2 - LLMOps"})
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

# Common tags/params for every run: what ran, where, and under which pricing assumptions
RUN_TAGS = {"provider": PROVIDER, "model": MODEL, "api_version": API_VERSION, "host": RUN_ENV["host"],
            "pricing_date": PRICING_DATE, "currency": CURRENCY}
print(f"tracking URI : {TRACKING_URI}\nexperiment   : {experiment.name} (id {experiment.experiment_id})")

In [ ]:
# =============================================================================
# Task 3.1b - Log the Task 1/2 baseline measurements as their own MLflow run
# Keeps latency/cost evidence from Tasks 1-2 in the same experiment as Task 3.
# =============================================================================
with mlflow.start_run(run_name="task1_task2_latency_cost_baseline", tags={**RUN_TAGS, "stage": "baseline"}):
    mlflow.log_params({"prompt_version": "v1", "n_requests": len(task2_df), "temperature": TEMPERATURE,
                       "max_output_tokens": MAX_OUTPUT_TOKENS, "input_price_per_1m": PRICE["input"],
                       "output_price_per_1m": PRICE["output"]})
    mlflow.log_metrics({"mean_latency_seconds": lat.mean(), "median_latency_seconds": lat.median(),
                        "min_latency_seconds": lat.min(), "max_latency_seconds": lat.max(),
                        "std_latency_seconds": lat.std(ddof=1), "avg_cost_per_request": avg_cost,
                        "projected_monthly_cost_usd": monthly})
    mlflow.log_artifact(str(OUTPUT_DIR / "latency_tokens_cost.csv"))
    mlflow.log_artifact(str(FIG_DIR / "task1_latency.png"), artifact_path="plots")
    mlflow.log_artifact(str(FIG_DIR / "task2_cost_per_request.png"), artifact_path="plots")
    mlflow.log_text(interpretation + "\n\n" + recommendation, "analysis.txt")
print("baseline run logged")

In [ ]:
# =============================================================================
# Task 3.2 - Register prompt v1, v2, v3 in the MLflow Prompt Registry
# register_prompt() creates version 1, 2, 3 of `eduassist-academic-qa`, each with a
# commit message and tags. The helper is idempotent: if an identical template is
# already registered (notebook re-run), that version is reused instead of
# creating a duplicate - the registry only grows when the prompt really changes.
# =============================================================================
def existing_versions(name: str) -> list:
    """All registered versions of prompt `name` (empty list if the prompt does not exist yet)."""
    versions, v = [], 1
    while (pv := mlflow.genai.load_prompt(name, version=v, allow_missing=True, link_to_model=False)) is not None:
        versions.append(pv)
        v += 1
    return versions


def register_if_changed(label: str, spec: dict):
    """Register `spec['template']` as a new version unless the same text is already in the registry."""
    for pv in existing_versions(PROMPT_NAME):
        if pv.template == spec["template"]:
            return pv, "reused"
    pv = mlflow.genai.register_prompt(
        name=PROMPT_NAME, template=spec["template"], commit_message=spec["commit_message"],
        tags={"label": label, "style": spec["style"], "author": "2025ae05220", "task": "eduassist-qa"})
    return pv, "registered"


registered = {}
for label, spec in PROMPTS.items():
    pv, action = register_if_changed(label, spec)
    registered[label] = pv
    print(f"{label}: {action:<10} -> {pv.uri}  | {pv.commit_message}")

pd.DataFrame([{"label": k, "registry_version": v.version, "uri": v.uri, "variables": sorted(v.variables),
               "commit_message": v.commit_message} for k, v in registered.items()])

In [ ]:
# =============================================================================
# Task 3.3 - Prompt diff evidence
# difflib.unified_diff shows exactly which lines changed between consecutive
# versions (the same comparison the MLflow UI shows in the prompt "Compare" view).
# Saved to outputs/prompt_diffs.txt and logged to MLflow in Task 3.7.
# =============================================================================
def prompt_diff(a: str, b: str) -> str:
    """Unified diff of the registered templates of versions a -> b."""
    va, vb = registered[a], registered[b]
    return "\n".join(difflib.unified_diff(va.template.splitlines(), vb.template.splitlines(),
                                          fromfile=f"{PROMPT_NAME} v{va.version} ({a})",
                                          tofile=f"{PROMPT_NAME} v{vb.version} ({b})", lineterm=""))


diff_text = prompt_diff("v1", "v2") + "\n\n" + prompt_diff("v2", "v3")
(OUTPUT_DIR / "prompt_diffs.txt").write_text(diff_text + "\n", encoding="utf-8")
print(diff_text)

In [ ]:
# =============================================================================
# Task 3.4 - Output checks and MLflow scorers
# * is_valid_json   : the raw response parses as a JSON object - no stripping of
#                     code fences or surrounding prose (the brief forbids them).
# * is_schema_valid : valid JSON AND exactly the keys answer/confidence/key_points,
#                     confidence in {high, medium, low}, key_points a list of strings.
# * keyword_coverage: share of the question's expected concepts mentioned (0-1),
#                     a cheap proxy for answer content quality.
# The @scorer wrappers let mlflow.genai.evaluate() score each response; boolean
# scorer values are aggregated by MLflow into a mean = pass rate.
# =============================================================================
def is_valid_json(text: str) -> bool:
    """True if the whole response is a JSON object."""
    try:
        return isinstance(json.loads(text), dict)
    except (json.JSONDecodeError, TypeError):
        return False


def is_schema_valid(text: str) -> bool:
    """True if the response is valid JSON that follows the EduAssist schema exactly."""
    if not is_valid_json(text):
        return False
    obj = json.loads(text)
    return (set(obj) == {"answer", "confidence", "key_points"}
            and isinstance(obj["answer"], str) and obj["answer"].strip() != ""
            and obj["confidence"] in {"high", "medium", "low"}
            and isinstance(obj["key_points"], list) and all(isinstance(k, str) for k in obj["key_points"]))


def coverage(text: str, keywords: list[str]) -> float:
    """Fraction of expected keywords present (case-insensitive; 'a|b' accepts either alternative)."""
    low = text.lower()
    return sum(any(alt in low for alt in kw.split("|")) for kw in keywords) / len(keywords)


@scorer
def valid_json(outputs: str) -> bool:
    """MLflow scorer: response is valid JSON (aggregated mean = valid JSON rate)."""
    return is_valid_json(outputs)


@scorer
def schema_valid(outputs: str) -> bool:
    """MLflow scorer: response matches the answer/confidence/key_points schema."""
    return is_schema_valid(outputs)


@scorer
def keyword_coverage(outputs: str, expectations: dict) -> float:
    """MLflow scorer: share of expected concepts mentioned in the response."""
    return coverage(outputs, expectations["expected_keywords"])


# Sanity checks of the scorers on hand-written examples
assert is_valid_json('{"answer": "x", "confidence": "high", "key_points": ["a"]}')
assert not is_valid_json('```json\n{"answer": "x"}\n```')          # fenced -> invalid by the brief's rules
assert not is_schema_valid('{"answer": "x", "confidence": "sure", "key_points": []}')
assert coverage("Uses sunlight and CO2", ["sunlight|light", "carbon dioxide|co2", "water"]) == 2 / 3
print("scorer self-tests passed")

In [ ]:
# =============================================================================
# Task 3.5 - Evaluate one prompt version inside its own MLflow run
# For version `label`:
#   1. load the prompt from the registry by URI (links the run to that version),
#   2. call the LLM on the five questions with the *registered* template,
#   3. compute per-request costs and checks,
#   4. run mlflow.genai.evaluate() on the collected (input, output, expectation)
#      rows - outputs are pre-computed, so no extra API calls are made,
#   5. log params, the five required metrics, extra metrics and artifacts.
# =============================================================================
def evaluate_prompt_version(label: str) -> tuple[pd.DataFrame, dict]:
    """Run, score and log prompt version `label`; return per-request rows and the metric summary."""
    tags = {**RUN_TAGS, "stage": "prompt_evaluation", "prompt_label": label}
    with mlflow.start_run(run_name=f"prompt_{label}_{PROMPTS[label]['style']}", tags=tags) as run:
        pv = mlflow.genai.load_prompt(registered[label].uri)          # registry -> template (+ run link)
        assert pv.template == PROMPTS[label]["template"]
        print(f"Running {pv.uri} on {len(QUESTIONS)} questions ...")
        df = add_costs(run_batch(label, pv.template))
        kw = {q["question_id"]: q["expected_keywords"] for q in QUESTIONS}
        df["valid_json"] = df["response_text"].map(is_valid_json)
        df["schema_valid"] = df["response_text"].map(is_schema_valid)
        df["keyword_coverage"] = [coverage(t, kw[q]) for t, q in zip(df["response_text"], df["question_id"])]
        df["registry_version"] = pv.version

        # MLflow GenAI evaluation over the recorded responses (creates one trace + assessments per row)
        eval_data = [{"inputs": {"question_id": q["question_id"], "question": q["question"]},
                      "outputs": text, "expectations": {"expected_keywords": q["expected_keywords"]}}
                     for q, text in zip(QUESTIONS, df["response_text"])]
        eval_result = mlflow.genai.evaluate(data=eval_data, scorers=[valid_json, schema_valid, keyword_coverage])

        ok_rows = df[df["status"] == "success"]
        summary = {
            "avg_latency_seconds": ok_rows["latency_seconds"].mean(),
            "avg_input_tokens": ok_rows["input_tokens"].mean(),
            "avg_output_tokens": ok_rows["output_tokens"].mean(),
            "avg_cost_per_request": ok_rows["total_cost_usd"].mean(),
            "valid_json": df["valid_json"].mean(),           # valid JSON responses / total responses
            "schema_valid_rate": df["schema_valid"].mean(),
            "keyword_coverage": df["keyword_coverage"].mean(),
            "median_latency_seconds": ok_rows["latency_seconds"].median(),
            "max_latency_seconds": ok_rows["latency_seconds"].max(),
            "avg_total_tokens": ok_rows["total_tokens"].mean(),
            "projected_monthly_cost_usd": ok_rows["total_cost_usd"].mean() * MONTHLY_REQUESTS,
            "success_rate": (df["status"] == "success").mean(),
        }
        mlflow.log_params({"prompt_name": PROMPT_NAME, "prompt_version": pv.version, "prompt_label": label,
                           "prompt_uri": pv.uri, "model": MODEL, "temperature": TEMPERATURE,
                           "max_output_tokens": MAX_OUTPUT_TOKENS, "n_questions": len(QUESTIONS),
                           "input_price_per_1m": PRICE["input"], "output_price_per_1m": PRICE["output"]})
        mlflow.log_metrics(summary)
        mlflow.log_text(pv.template, f"prompt_{label}.txt")
        mlflow.log_table(df[["question_id", "latency_seconds", "input_tokens", "output_tokens", "total_cost_usd",
                             "valid_json", "schema_valid", "keyword_coverage", "response_text"]],
                         artifact_file=f"responses_{label}.json")
        df["mlflow_run_id"] = run.info.run_id
        print(f"  MLflow evaluate metrics: { {k: round(float(v), 3) for k, v in eval_result.metrics.items()} }")
    return df, summary

In [ ]:
# =============================================================================
# Task 3.6 - Run all three prompt versions on the same five questions
# Produces three MLflow runs and the per-request detail table
# (outputs/prompt_evaluation_details.csv - one row per version x question).
# =============================================================================
details, summaries = [], {}
for label in PROMPTS:
    df_v, summary_v = evaluate_prompt_version(label)
    details.append(df_v)
    summaries[label] = summary_v
    time.sleep(REQUEST_INTERVAL_S)

details_df = pd.concat(details, ignore_index=True)
DETAIL_COLS = ["prompt_version", "registry_version", "question_id", "model", "start_time", "end_time",
               "latency_seconds", "status", "input_tokens", "output_tokens", "total_tokens", "total_cost_usd",
               "valid_json", "schema_valid", "keyword_coverage", "finish_reason", "mlflow_run_id", "response_text"]
details_df[DETAIL_COLS].to_csv(OUTPUT_DIR / "prompt_evaluation_details.csv", index=False)
details_df[DETAIL_COLS[:-2]]

In [ ]:
# =============================================================================
# Task 3.7 - Compare versions and choose the production prompt
# Decision rule (stated before looking at the numbers):
#   1. Gate - the app parses the reply as JSON, so only versions with valid_json = 1.0
#      (and schema-valid) can go to production; if none passes, the highest
#      valid_json rate wins.
#   2. Among eligible versions: higher keyword coverage (content quality), then lower
#      average cost, then lower average latency.
# The comparison table is the deliverable outputs/prompt_evaluation_results.csv.
# =============================================================================
comparison = pd.DataFrame(summaries).T.rename_axis("prompt_version").reset_index()
comparison["registry_version"] = comparison["prompt_version"].map(lambda k: registered[k].version)

eligible = comparison[(comparison["valid_json"] >= 1.0) & (comparison["schema_valid_rate"] >= 1.0)]
pool = eligible if not eligible.empty else comparison[comparison["valid_json"] == comparison["valid_json"].max()]
best = pool.sort_values(["keyword_coverage", "avg_cost_per_request", "avg_latency_seconds"],
                        ascending=[False, True, True]).iloc[0]
BEST_LABEL = best["prompt_version"]
BEST_VERSION = int(best["registry_version"])
comparison["recommended"] = np.where(comparison["prompt_version"] == BEST_LABEL, "Yes", "No")

RESULT_COLS = ["prompt_version", "registry_version", "avg_latency_seconds", "avg_input_tokens", "avg_output_tokens",
               "avg_cost_per_request", "valid_json", "schema_valid_rate", "keyword_coverage",
               "projected_monthly_cost_usd", "recommended"]
comparison[RESULT_COLS].to_csv(OUTPUT_DIR / "prompt_evaluation_results.csv", index=False)

# Formatted view = the "Prompt comparison table" of the brief
comparison[RESULT_COLS].style.format({
    "avg_latency_seconds": "{:.2f} s", "avg_input_tokens": "{:.1f}", "avg_output_tokens": "{:.1f}",
    "avg_cost_per_request": "${:.6f}", "valid_json": "{:.0%}", "schema_valid_rate": "{:.0%}",
    "keyword_coverage": "{:.0%}", "projected_monthly_cost_usd": "${:.2f}"}).hide(axis="index")

In [ ]:
# =============================================================================
# Task 3.8 - Comparison charts (small multiples, one metric per panel, one axis each)
# Same colour per prompt version in every panel; values are labelled directly.
# =============================================================================
panels = [("avg_latency_seconds", "Avg latency (s)", "{:.2f}"),
          ("avg_output_tokens", "Avg output tokens", "{:.0f}"),
          ("avg_cost_per_request", "Avg cost / request (micro-USD)", "{:.0f}"),
          ("valid_json", "Valid JSON rate (%)", "{:.0f}")]
scale = {"avg_cost_per_request": 1e6, "valid_json": 100}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (col, title, fmt) in zip(axes, panels):
    vals = comparison[col].astype(float) * scale.get(col, 1)
    bars = ax.bar(comparison["prompt_version"], vals, width=0.6,
                  color=[SERIES[v] for v in comparison["prompt_version"]])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v), ha="center", va="bottom", fontsize=9, color=INK)
    style_axes(ax, title, "")
    ax.set_ylim(0, max(vals.max() * 1.25, 1))
fig.suptitle(f"Task 3 - prompt version comparison ({MODEL}); recommended: {BEST_LABEL}", x=0.01, ha="left",
             fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig(FIG_DIR / "task3_prompt_comparison.png", dpi=150)
display(Image(str(FIG_DIR / "task3_prompt_comparison.png")))

In [ ]:
# =============================================================================
# Task 3.9 - Promote the best version: MLflow alias "production"
# set_prompt_alias() moves the alias to the chosen registry version; the app then
# loads "prompts:/eduassist-academic-qa@production" and never hard-codes a version.
# Rolling back = pointing the alias at another version - no code change.
# A version tag records the decision, and the alias is verified by loading it back.
# =============================================================================
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=BEST_VERSION)
mlflow.genai.set_prompt_version_tag(PROMPT_NAME, BEST_VERSION, "status", "production")
mlflow.genai.set_prompt_version_tag(PROMPT_NAME, BEST_VERSION, "promoted_on", datetime.now(timezone.utc).date().isoformat())

prod = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production", link_to_model=False)
assert prod.version == BEST_VERSION and prod.template == PROMPTS[BEST_LABEL]["template"]
print(f"alias 'production' -> {PROMPT_NAME} version {prod.version} ({BEST_LABEL}); aliases on it: {prod.aliases}")
print("\nproduction template:\n" + prod.format(question="<student question>"))

In [ ]:
# =============================================================================
# Task 3.10 - Promotion justification (>= 150 words) built from the measured metrics
# Covers quality, JSON validity, latency, tokens and cost, then logs the decision
# run (comparison table, chart, diffs, justification) to MLflow.
# =============================================================================
c = comparison.set_index("prompt_version")
b = c.loc[BEST_LABEL]
others_lbl = [v for v in c.index if v != BEST_LABEL]


def fmt_v(label: str) -> str:
    """One-line metric summary of a version for the justification text."""
    r = c.loc[label]
    return (f"{label} ({r.avg_latency_seconds:.2f}s, {r.avg_input_tokens:.0f} in / {r.avg_output_tokens:.0f} out tokens, "
            f"${r.avg_cost_per_request:.6f} per request, valid JSON {r.valid_json:.0%})")


best_other_cov = max(c.loc[o].keyword_coverage for o in others_lbl)
if b.keyword_coverage + 0.05 < best_other_cov:   # promoted version covers noticeably fewer concepts
    quality_note = (f"The longer free-text answers cover more concepts, but {BEST_LABEL}'s compact answer still "
                    f"states the core idea, and a more complete answer the app cannot parse has no production value - "
                    f"the coverage gap is an accepted, monitored trade-off.")
else:
    quality_note = "So the structured format costs no content quality relative to the free-text prompts."

justification = f"""Promotion decision: {PROMPT_NAME} version {BEST_VERSION} ({BEST_LABEL}, {PROMPTS[BEST_LABEL]['style']}) is promoted to the `production` alias.

JSON validity. EduAssist's application parses every reply as JSON with the fields answer, confidence and key_points, so format reliability is a hard gate rather than a nice-to-have. {BEST_LABEL} returned valid JSON for {b.valid_json:.0%} of the five questions and matched the full schema for {b.schema_valid_rate:.0%}, while {', '.join(f'{o} reached {c.loc[o].valid_json:.0%}' for o in others_lbl)}: free-text or markdown answers would break the parser and require fragile post-processing.

Quality. On content, {BEST_LABEL} mentioned {b.keyword_coverage:.0%} of the expected concepts per question ({', '.join(f'{o}: {c.loc[o].keyword_coverage:.0%}' for o in others_lbl)}). {quality_note} The key_points list and the self-reported confidence level also give the front end structured material that free-text answers cannot provide.

Latency and tokens. Measured on the same five questions with {MODEL}: {fmt_v(BEST_LABEL)} versus {'; '.join(fmt_v(o) for o in others_lbl)}. The JSON prompt is longer on the input side, but input tokens are cheap and are processed in parallel; latency is dominated by output length, and the schema keeps answers compact.

Cost. At ${PRICE['input']:.2f} / ${PRICE['output']:.2f} per million input/output tokens (pricing date {PRICING_DATE}, {CURRENCY}), {BEST_LABEL} projects to ${b.projected_monthly_cost_usd:.2f} for 10,000 requests per month, against {', '.join(f'${c.loc[o].projected_monthly_cost_usd:.2f} for {o}' for o in others_lbl)}. Because output tokens cost {PRICE['output'] / PRICE['input']:.1f}x more than input tokens, the shorter structured answer outweighs the extra prompt tokens.

Risks and next steps. Five questions at temperature 0 are a small sample; before a wider rollout the evaluation set should grow, valid_json should be monitored in production, and a strict parser with a retry should guard the rare invalid reply. Because the app loads prompts:/{PROMPT_NAME}@production, a rollback is a single alias change."""

n_words = len(justification.split())
assert n_words >= 150, n_words
(OUTPUT_DIR / "promotion_justification.md").write_text(justification + "\n", encoding="utf-8")
display(Markdown(justification))
print(f"({n_words} words)")

# Decision run: everything a reviewer needs to audit the promotion in one place
with mlflow.start_run(run_name="promotion_decision", tags={**RUN_TAGS, "stage": "promotion"}):
    mlflow.log_params({"prompt_name": PROMPT_NAME, "promoted_label": BEST_LABEL, "promoted_version": BEST_VERSION,
                       "alias": "production", "decision_rule": "valid_json=1 gate, then coverage, cost, latency"})
    mlflow.log_metrics({f"{lbl}_{m}": float(c.loc[lbl, m]) for lbl in c.index
                        for m in ["avg_latency_seconds", "avg_cost_per_request", "valid_json", "keyword_coverage"]})
    for f in ["prompt_evaluation_results.csv", "prompt_evaluation_details.csv", "prompt_diffs.txt",
              "promotion_justification.md"]:
        mlflow.log_artifact(str(OUTPUT_DIR / f))
    mlflow.log_artifact(str(FIG_DIR / "task3_prompt_comparison.png"), artifact_path="plots")
print("promotion_decision run logged")

## 4. Run summary and evidence checklist

In [ ]:
# =============================================================================
# Section 4 - Summary of generated files and the MLflow evidence to capture
# Lists every deliverable with its size and prints the commands/screens used for
# the screenshots in LAB_RUNBOOK.md.
# =============================================================================
files = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())
display(pd.DataFrame({"file": [str(p.relative_to(ROOT)) for p in files],
                      "bytes": [p.stat().st_size for p in files]}))
print(f"Host: {RUN_ENV['host']} | model: {MODEL} | promoted: {PROMPT_NAME} v{BEST_VERSION} ({BEST_LABEL}) -> production")
print("MLflow UI:  mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000   (from the repo root)")
print("Screens:    Experiments > eduassist_llmops_assignment (runs + metrics) | Prompts > eduassist-academic-qa "
      "(versions, Compare = diff, alias production)")